In [1]:
import os
from dotenv import load_dotenv

load_dotenv() 

COINGECKO_API_KEY = os.environ["COINGECKO_API_KEY"]
DATABASE_URL = os.environ["DATABASE_URL"]

In [5]:
import time
import requests
import pandas as pd
from Coin_mapping import COIN_TO_BINANCE

headers = {
    "accept": "application/json",
    "x-cg-demo-api-key": COINGECKO_API_KEY
}

coins_map = {k: v for k, v in COIN_TO_BINANCE.items() if v is not None}

all_rows = []
print(f"Starting historical backfill for {len(coins_map)} coins, last 30 days...")

for cg_id, binance_id in coins_map.items():
    print(f"Fetching {cg_id} ({binance_id})....")
    cg_url = f"https://api.coingecko.com/api/v3/coins/{cg_id}/market_chart?vs_currency=usd&days=30&interval=daily"
    cg_res = requests.get(cg_url, headers=headers)
    if cg_res.status_code != 200:
        print(f" CoinGecko failed for {cg_id}: HTTP {cg_res.status_code} - skipping this coin")
        continue
    cg_data = cg_res.json()

    cg_df = pd.DataFrame({
        'timestamp': [x[0] for x in cg_data['prices']],
        'cg_price': [x[1] for x in cg_data['prices']],
        'cg_volume_usd': [x[1] for x in cg_data['total_volumes']],
        'market_cap_usd': [x[1] for x in cg_data['market_caps']]
    })
    cg_df['date'] = pd.to_datetime(cg_df['timestamp'], unit='ms').dt.date
    cg_df = cg_df.drop_duplicates(subset=['date']).drop(columns=['timestamp'])

    binance_url = f"https://api.binance.com/api/v3/klines?symbol={binance_id}&interval=1d&limit=31"
    binance_res = requests.get(binance_url)
    if binance_res.status_code != 200:
        print(f"  Binance failed for {binance_id}: HTTP {binance_res.status_code} - skipping this coin")
        continue
    binance_raw = binance_res.json()

    b_df = pd.DataFrame(binance_raw)
    binance_df = pd.DataFrame({
        'date': pd.to_datetime(b_df[0], unit='ms').dt.date,
        'open_price': b_df[1].astype(float),
        'high_price': b_df[2].astype(float),
        'low_price': b_df[3].astype(float),
        'binance_price': b_df[1].astype(float),
        'binance_volume_usd': b_df[7].astype(float)
    }).drop_duplicates(subset=['date'])

    merged_df = pd.merge(cg_df, binance_df, on='date', how='inner')
    merged_df['coingecko_id'] = cg_id
    merged_df['binance_id'] = binance_id
    merged_df = merged_df.sort_values('date', ascending=True).reset_index(drop=True)

    # Feature engineering - unchanged from what you wrote, it was good
    merged_df['spread_usd'] = merged_df['binance_price'] - merged_df['cg_price']
    merged_df['price_diff_pct'] = (abs(merged_df['spread_usd']) / merged_df['cg_price']) * 100
    merged_df['premium_exchange'] = merged_df['spread_usd'].apply(lambda x: 'binance' if x > 0 else 'coingecko')
    merged_df['daily_return_pct'] = merged_df['binance_price'].pct_change() * 100
    merged_df['volatility_7d'] = merged_df['daily_return_pct'].rolling(window=7, min_periods=1).std().fillna(0)
    merged_df['daily_range_pct'] = ((merged_df['high_price'] - merged_df['low_price']) / merged_df['binance_price']) * 100
    merged_df['binance_volume_share_pct'] = (merged_df['binance_volume_usd'] / merged_df['cg_volume_usd']) * 100
    merged_df['is_arbitrage_viable'] = merged_df['price_diff_pct'] > 0.35

    all_rows.append(merged_df)
    time.sleep(1)

full_df = pd.concat(all_rows, ignore_index=True)
print(f"Built {len(full_df)} total rows across {len(coins_map)} coins.")

# pandas' NaN isn't the same thing as SQL NULL. If you don't convert
# it, psycopg2 either errors or silently writes garbage. This line converts
# every NaN in the DataFrame to a real Python None first.
full_df = full_df.astype(object).where(pd.notnull(full_df), None)
print("Extraction, merge, and feature engineering complete.")


Starting historical backfill for 19 coins, last 30 days...
Fetching bitcoin (BTCUSDT)....
Fetching ethereum (ETHUSDT)....
Fetching binancecoin (BNBUSDT)....
Fetching solana (SOLUSDT)....
Fetching ripple (XRPUSDT)....
Fetching usd-coin (USDCUSDT)....
Fetching cardano (ADAUSDT)....
Fetching dogecoin (DOGEUSDT)....
Fetching tron (TRXUSDT)....
Fetching avalanche-2 (AVAXUSDT)....
Fetching polkadot (DOTUSDT)....
Fetching chainlink (LINKUSDT)....
Fetching matic-network (MATICUSDT)....
Fetching litecoin (LTCUSDT)....
Fetching shiba-inu (SHIBUSDT)....
Fetching bitcoin-cash (BCHUSDT)....
Fetching uniswap (UNIUSDT)....
Fetching stellar (XLMUSDT)....
Fetching cosmos (ATOMUSDT)....
Built 540 total rows across 19 coins.
Extraction, merge, and feature engineering complete.


In [6]:
import psycopg2

conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

upsert_query = """
    INSERT INTO daily_price_metrics (
        date, coingecko_id, binance_id, cg_price, cg_volume_usd, market_cap_usd,
        binance_price, open_price, high_price, low_price, binance_volume_usd,
        spread_usd, price_diff_pct, premium_exchange, daily_return_pct,
        volatility_7d, daily_range_pct, binance_volume_share_pct, is_arbitrage_viable
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    ON CONFLICT (date, coingecko_id)
    DO UPDATE SET
        cg_price = EXCLUDED.cg_price,
        binance_price = EXCLUDED.binance_price,
        spread_usd = EXCLUDED.spread_usd,
        price_diff_pct = EXCLUDED.price_diff_pct,
        is_arbitrage_viable = EXCLUDED.is_arbitrage_viable;
"""

for _, row in full_df.iterrows():
    cursor.execute(upsert_query, (
        row['date'], row['coingecko_id'], row['binance_id'], row['cg_price'],
        row['cg_volume_usd'], row['market_cap_usd'], row['binance_price'],
        row['open_price'], row['high_price'], row['low_price'], row['binance_volume_usd'],
        row['spread_usd'], row['price_diff_pct'], row['premium_exchange'],
        row['daily_return_pct'], row['volatility_7d'], row['daily_range_pct'],
        row['binance_volume_share_pct'], row['is_arbitrage_viable']
    ))

conn.commit()
cursor.close()
conn.close()
print(f"Saved {len(full_df)} rows to daily_price_metrics.")

Saved 540 rows to daily_price_metrics.
